In [3]:
import pandas as pd
from pathlib import Path

#Loading final processed file

path = Path("../data/processed")
movies = pd.read_csv(path/"movies_final.csv")
movies.shape

(5798, 36)

In [4]:
#ML imports

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
movies['combined_features'].isnull().sum()

np.int64(0)

In [9]:
#initialize tf-idf

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = vectorizer.fit_transform(movies['combined_features'])
tfidf_matrix.shape

(5798, 5000)

In [14]:
tfidf_matrix[0].toarray()

array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 5000))

In [15]:
#similarity matrix

similarity_matrix = cosine_similarity(tfidf_matrix)
similarity_matrix.shape

(5798, 5798)

In [18]:
#recommender function

def recommend_movies(title, top_n=5):
    title = title.lower()
    
    if title not in movies["title"].str.lower().values:
        return "Movie not found."
    
    idx = movies[movies["title"].str.lower() == title].index[0]
    
    similarity_scores = list(enumerate(similarity_matrix[idx]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    
    top_movies = similarity_scores[1:top_n+1]
    
    movie_indices = [i[0] for i in top_movies]
    scores = [i[1] for i in top_movies]
    
    results = movies.iloc[movie_indices][["title", "vote_average"]].copy()
    results["similarity_score"] = scores
    
    return results

In [19]:
recommend_movies("Avatar")

,title,vote_average,similarity_score
5366,avatar: the way of water,7.651,0.327059
1245,colombiana,6.500,0.167141
942,the book of life,7.300,0.155708
3604,apollo 18,5.000,0.155645
2403,aliens,7.700,0.150219


In [20]:
#saving model artificats

import pickle

# Save vectorizer
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

# Save similarity matrix
with open("similarity_matrix.pkl", "wb") as f:
    pickle.dump(similarity_matrix, f)

# Save movies metadata
with open("movies_metadata.pkl", "wb") as f:
    pickle.dump(movies, f)